In [1]:
!pip install -q transformers torch sentencepiece


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

C:\Users\athar\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_id = "abhi099k/ai-text-detector-v-n4.0"

print(f"Downloading and loading {model_id}...")
# Load tokenizer and sequence classification model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

# Automatically move the model to GPU for faster inference if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model successfully loaded on: {device}")

Model successfully loaded on: cuda


In [4]:
# Cell 4: Define the prediction function
def detect_ai_text(text):
    # Tokenize input and truncate to the model's max length (512 tokens)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    
    # Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Perform inference without calculating gradients
    with torch.no_grad():
        outputs = model(**inputs)
        
    # Apply softmax to logits to get percentage probabilities
    probabilities = F.softmax(outputs.logits, dim=1)
    
    # For this specific model: Class 0 is Human, Class 1 is AI
    predicted_class = torch.argmax(probabilities, dim=1).item()
    human_prob = probabilities[0][0].item() * 100
    ai_prob = probabilities[0][1].item() * 100
    
    result = "🤖 AI-Generated" if predicted_class == 1 else "🧑 Human-Written"
    
    return {
        "prediction": result,
        "ai_probability": f"{ai_prob:.2f}%",
        "human_probability": f"{human_prob:.2f}%"
    }

In [5]:
# Cell 5: Test custom text
custom_text = """
 THE CONSTITUTION OF INDIA
                                (Part V.—The Union)
       (2) No person shall be a member both of Parliament and of a House of
the Legislature of a State 1***, and if a person is chosen a member both of
Parliament and of a House of the Legislature of 2[a State], then, at the
expiration of such period as may be specified in rules made by the President,
that person’s seat in Parliament shall become vacant, unless he has previously
resigned his seat in the Legislature of the State.
       (3) If a member of either House of Parliament—
            (a) becomes subject to any of the disqualifications mentioned in
       3
         [clause (1) or clause (2) of article 102]; or
            4
              [(b) resigns his seat by writing under his hand addressed to the
       Chairman or the Speaker, as the case may be, and his resignation is
       accepted by the Chairman or the Speaker, as the case may be,]
his seat shall thereupon become vacant:
       5
           [Provided that in the case of any resignation referred to in sub-clause (b),
if from information received or otherwise and after making such inquiry as he
thinks fit, the Chairman or the Speaker, as the case may be, is satisfied that
such resignation is not voluntary or genuine, he shall not accept such
resignation.]
       (4) If for a period of sixty days a member of either House of Parliament
is without permission of the House absent from all meetings thereof, the House
may declare his seat vacant:
       Provided that in computing the said period of sixty days no account shall
be taken of any period during which the House is prorogued or is adjourned for
more than four consecutive days.

"""

print("Analyzing text...\n")
results = detect_ai_text(custom_text)

print(f"Input Text:\n\"{custom_text.strip()}\"\n")
print("-" * 40)
print(f"Final Prediction:  {results['prediction']}")
print(f"AI Likelihood:     {results['ai_probability']}")
print(f"Human Likelihood:  {results['human_probability']}")

Analyzing text...

Input Text:
"THE CONSTITUTION OF INDIA
                                (Part V.—The Union)
       (2) No person shall be a member both of Parliament and of a House of
the Legislature of a State 1***, and if a person is chosen a member both of
Parliament and of a House of the Legislature of 2[a State], then, at the
expiration of such period as may be specified in rules made by the President,
that person’s seat in Parliament shall become vacant, unless he has previously
resigned his seat in the Legislature of the State.
       (3) If a member of either House of Parliament—
            (a) becomes subject to any of the disqualifications mentioned in
       3
         [clause (1) or clause (2) of article 102]; or
            4
              [(b) resigns his seat by writing under his hand addressed to the
       Chairman or the Speaker, as the case may be, and his resignation is
       accepted by the Chairman or the Speaker, as the case may be,]
his seat shall thereupon 